## Data Access
This notebook, `03_data_access.ipynb` demonstrates how to load the fully
processed, high-quality single-cell feature matrix. It requires a high-memory machine.

In [1]:
import anndata as ad
import pickle
import pandas as pd
import scmorph as sm

file = "workspace/profiles_assembled/profiles_processed/all_adata_featFilt_batchCorr.h5ad"

# Save memory
use_backed = False

adata = sm.read(file, backed="r" if use_backed else False)

/exports/igmm/eddie/khamseh-lab/jwagner/mamba/envs/micromorph/lib/python3.10/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [2]:
# load dependencies
import pandas as pd
import numpy as np
import scmorph as sm
from tqdm import tqdm
from plotnine import *
import matplotlib.pyplot as plt
from natsort import natsorted

In [3]:
adatas_per_plate = {}
for plate in adata.obs["PlateID"].unique():
    adata_ss = adata[adata.obs["PlateID"] == plate]
    adatas_per_plate[plate] = adata_ss

In [4]:
# Define control wells present across all plates
ctrl_wells = ['E23', 'E24', 'F23', 'F24', 'I23', 'I24', 'J23', 'J24']

# Test treatments' significance per plate
ref_ks_d = {}
treat_ks_d = {}
for plate in tqdm(adatas_per_plate):
    ref_ks_d[plate] = {}
    treat_ks_d[plate] = {}
    adata_ss = adatas_per_plate[plate]
    if adata_ss.isbacked:
        adata_ss = adata_ss.to_memory()
    ref_ks_d[plate], treat_ks_d[plate] = sm.tl.get_ks(adata_ss,
                                                        treatment_key="treatment",
                                                        control_wells=ctrl_wells,
                                                        progress=False,
                                                        n_pcs=10,
                                                        scale_by_var=True)

100%|██████████| 134/134 [07:47<00:00,  3.49s/it]


In [6]:
treat_ks_d[plate]

,control,treatment,ks_stat,ks_pval,ks_qval,is_significant_0.05,is_significant_0.1
0,control_neg_mimic,DMSO,0.109051,3.380659e-04,8.958747e-04,False,False
1,control_neg_mimic,control_cell_death,0.058353,1.397987e-04,4.490505e-04,False,False
2,control_neg_mimic,control_tr_rotenone,0.001539,9.937746e-01,9.937746e-01,False,False
3,control_neg_mimic,control_transfection_reagent_only,0.009778,7.589196e-01,8.650051e-01,False,False
4,control_neg_mimic,empty,0.035224,6.198346e-04,1.428314e-03,False,False
...,...,...,...,...,...,...,...
101,control_neg_mimic,hsa-miR-8069,0.090189,2.188672e-04,6.557334e-04,False,False
102,control_neg_mimic,hsa-miR-8072,0.109576,1.595548e-05,7.047004e-05,False,False
103,control_neg_mimic,hsa-miR-8073,0.123052,1.160232e-07,8.784616e-07,False,True
104,control_neg_mimic,hsa-miR-8078,0.047581,2.488801e-01,3.178469e-01,False,False


In [18]:
plateid_to_meta = adata.obs.set_index('PlateID')[["PlateLayout",'CellLine', 'Replicate']].drop_duplicates().apply(list, axis=1).to_dict()

In [ ]:
for plate in treat_ks_d:
    plate_layout, cell_line, rep = plateid_to_meta[plate]
    ref_ks_d[plate].insert(0, "PlateLayout", plate_layout)
    ref_ks_d[plate].insert(0, "Replicate", rep)
    ref_ks_d[plate].insert(0, "CellLine", cell_line)
    ref_ks_d[plate].insert(0, "PlateID", plate)
    treat_ks_d[plate].insert(0, "PlateLayout", plate_layout)
    treat_ks_d[plate].insert(0, "Replicate", rep)
    treat_ks_d[plate].insert(0, "CellLine", cell_line)
    treat_ks_d[plate].insert(0, "PlateID", plate)

In [21]:
from processing_helper import map_cell_lines

In [29]:
ref_ks=pd.concat(ref_ks_d).dropna()
treat_ks=pd.concat(treat_ks_d).dropna()
map_cell_lines(ref_ks)
map_cell_lines(treat_ks)

In [ ]:
treat_ks.reset_index(drop=True).to_csv("output/all_hit_calling_results.csv",index=False)